# **YOLOv5**
*   github程式碼: https://github.com/ultralytics/yolov5
*   github模型權重: https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt
---
1.   載入YOLOv5程式及環境基本配置
2.   命令列執行圖像物件偵測
3.   .ipynb執行圖像物件偵測
4.   命令列執行影片物件偵測








# **載入YOLOv5程式及環境基本配置**

下載程式碼及安裝環境

In [ ]:
!git clone https://github.com/ultralytics/yolov5  # clone
%cd yolov5
%pip install -qr requirements.txt comet_ml  # install

import torch
import utils
display = utils.notebook_init()  # checks

# **命令列執行圖像物件偵測**

資料夾圖片預測並將預測結果存檔

In [ ]:
!python detect.py --weights yolov5s.pt --img 640 --conf 0.25 --source data/images

顯示圖片預測結果

In [ ]:
display.Image(filename='runs/detect/exp/zidane.jpg', width=600)

# **.ipynb執行圖像物件偵測**

載入library

In [ ]:
import argparse
import csv
import os
import platform
import sys
from pathlib import Path

import torch

from ultralytics.utils.plotting import Annotator, colors, save_one_box

from models.common import DetectMultiBackend
from utils.dataloaders import IMG_FORMATS, VID_FORMATS, LoadImages, LoadScreenshots, LoadStreams
from utils.general import (
    LOGGER,
    Profile,
    check_file,
    check_img_size,
    check_imshow,
    check_requirements,
    colorstr,
    cv2,
    increment_path,
    non_max_suppression,
    print_args,
    scale_boxes,
    strip_optimizer,
    xyxy2xywh,
)
from utils.torch_utils import select_device, smart_inference_mode

物件偵測參數
*   'weights'：模型的路徑或 Triton URL。
*   'source'：資料來源的路徑，可以是檔案、目錄、URL、glob、螢幕或網路攝影機。
*   'data'：資料集的配置文件路徑。
*   'imgsz'：推論的圖像大小（高度，寬度）。
*   'conf_thres'：信心閾值，用於過濾低信心的預測。
*   'iou_thres'：NMS 的 IOU 閾值，用於過濾重疊的預測。
*   'max_det'：每張圖像的最大檢測數量。
*   'device'：CUDA 裝置，例如 “0” 或 “0,1,2,3” 或 “cpu”。
*   'view_img'：是否顯示結果。(在colab建議不使用)
*   'save_txt'：是否將結果儲存為 *.txt。
*   'save_csv'：是否將結果儲存為 CSV 格式。
*   'save_conf'：是否在 --save-txt 標籤中儲存信心度。
*   'save_crop'：是否儲存裁剪的預測框。
*   'nosave'：是否不儲存圖像/影片。
*   'classes'：過濾類別，例如 --class 0 或 --class 0 2 3。
*   'agnostic_nms'：是否使用類別不明確的 NMS。
*   'augment'：是否使用增強推論。
*   'visualize'：是否視覺化特徵。
*   'update'：是否更新所有模型。
*   'project'：儲存結果的專案名稱。
*   'name'：儲存結果的名稱。
*   'exist_ok'：如果專案/名稱已存在，則不增加。
*   'line_thickness'：邊界框的厚度（像素）。
*   'hide_labels'：是否隱藏標籤。
*   'hide_conf'：是否隱藏信心度。
*   'half'：是否使用 FP16 半精度推論。
*   'dnn'：是否使用 OpenCV DNN 進行 ONNX 推論。
*   'vid_stride'：影片的幀率步長。

In [ ]:
args = {
    'weights': "yolov5s.pt",  # model path or triton URL
    'source': "data/images",  # file/dir/URL/glob/screen/0(webcam)
    'data': "data/coco128.yaml",  # dataset.yaml path
    'imgsz': (640, 640),  # inference size (height, width)
    'conf_thres': 0.25,  # confidence threshold
    'iou_thres': 0.45,  # NMS IOU threshold
    'max_det': 1000,  # maximum detections per image
    'device': "",  # cuda device, i.e. 0 or 0,1,2,3 or cpu
    'view_img': False,  # show results
    'save_txt': False,  # save results to *.txt
    'save_csv': False,  # save results in CSV format
    'save_conf': False,  # save confidences in --save-txt labels
    'save_crop': False,  # save cropped prediction boxes
    'nosave': False,  # do not save images/videos
    'classes': None,  # filter by class: --class 0, or --class 0 2 3
    'agnostic_nms': False,  # class-agnostic NMS
    'augment': False,  # augmented inference
    'visualize': False,  # visualize features
    'update': False,  # update all models
    'project': "runs/detect",  # save results to project/name
    'name': "exp",  # save results to project/name
    'exist_ok': False,  # existing project/name ok, do not increment
    'line_thickness': 3,  # bounding box thickness (pixels)
    'hide_labels': False,  # hide labels
    'hide_conf': False,  # hide confidences
    'half': False,  # use FP16 half-precision inference
    'dnn': False,  # use OpenCV DNN for ONNX inference
    'vid_stride': 1,  # video frame-rate stride
}

取得要使用物件偵測處理的資料類別

In [ ]:
source = str(args["source"])
save_img = not args["nosave"] and not source.endswith(".txt")  # save inference images
is_file = Path(source).suffix[1:] in (IMG_FORMATS + VID_FORMATS)
is_url = source.lower().startswith(("rtsp://", "rtmp://", "http://", "https://"))
webcam = source.isnumeric() or source.endswith(".streams") or (is_url and not is_file)
screenshot = source.lower().startswith("screen")
if is_url and is_file:
  source = check_file(source)  # download

建立結果輸出資料夾

In [ ]:
# Directories
save_dir = increment_path(Path(args["project"]) / args["name"], exist_ok=args["exist_ok"])  # increment run
(save_dir / "labels" if args["save_txt"] else save_dir).mkdir(parents=True, exist_ok=True)  # make dir

模型載入權重，恢復訓練好的模型以用來測試

In [ ]:
# Load model
device = select_device(args["device"])
model = DetectMultiBackend(args["weights"], device=device, dnn=args["dnn"], data=args["data"], fp16=args["half"])
stride, names, pt = model.stride, model.names, model.pt
imgsz = check_img_size(args["imgsz"], s=stride)  # check image size

根據資料集類別使用對應的資料集處理方式

In [ ]:
# Dataloader
bs = 1  # batch_size
if webcam:
  view_img = check_imshow(warn=True)
  dataset = LoadStreams(source, img_size=imgsz, stride=stride, auto=pt, vid_stride=args["vid_stride"])
  bs = len(dataset)
elif screenshot:
  dataset = LoadScreenshots(source, img_size=imgsz, stride=stride, auto=pt)
else:
  dataset = LoadImages(source, img_size=imgsz, stride=stride, auto=pt, vid_stride=args["vid_stride"])
vid_path, vid_writer = [None] * bs, [None] * bs

模型預熱及參數初始化

In [ ]:
# Run inference
model.warmup(imgsz=(1 if pt or model.triton else bs, 3, *imgsz))  # warmup
seen, windows, dt = 0, [], (Profile(device=device), Profile(device=device), Profile(device=device))

讀取資料集的內容做預測及輸出預測結果
*   每個來源(ex.圖片、影片、路徑)=>模型=>預測結果
    1.   資料轉成模型輸入張量
    2.   模型預測結果
    3.   非最大抑制，以消除重疊的預測框。
*   每個來源的每張圖片根據預測結果儲存預測結果
    1.   儲存圖片預測結果(ex.csv每列、每個txt檔案每列、annotator每個box_label、每個框選位置圖片)
    2.   圖片畫標記及顯示標記圖片
    3.   儲存標記圖片或影片串流





In [ ]:
for path, im, im0s, vid_cap, s in dataset:
  with dt[0]:
    im = torch.from_numpy(im).to(model.device)
    im = im.half() if model.fp16 else im.float()  # uint8 to fp16/32
    im /= 255  # 0 - 255 to 0.0 - 1.0
    if len(im.shape) == 3:
      im = im[None]  # expand for batch dim
    if model.xml and im.shape[0] > 1:
      ims = torch.chunk(im, im.shape[0], 0)
  # Inference
  with dt[1]:
    visualize = increment_path(save_dir / Path(path).stem, mkdir=True) if args["visualize"] else False
    if model.xml and im.shape[0] > 1:
      pred = None
      for image in ims:
        if pred is None:
          pred = model(image, augment=args["augment"], visualize=visualize).unsqueeze(0)
        else:
          pred = torch.cat((pred, model(image, augment=args["augment"], visualize=visualize).unsqueeze(0)), dim=0)
      pred = [pred, None]
    else:
      pred = model(im, augment=args["augment"], visualize=visualize)
  # NMS
  with dt[2]:
    pred = non_max_suppression(pred, args["conf_thres"], args["iou_thres"], args["classes"], args["agnostic_nms"], max_det=args["max_det"])

  # Second-stage classifier (optional)
  # pred = utils.general.apply_classifier(pred, classifier_model, im, im0s)
  # Define the path for the CSV file
  csv_path = save_dir / "predictions.csv"
  # Create or append to the CSV file
  def write_to_csv(image_name, prediction, confidence):
    """Writes prediction data for an image to a CSV file, appending if the file exists."""
    data = {"Image Name": image_name, "Prediction": prediction, "Confidence": confidence}
    with open(csv_path, mode="a", newline="") as f:
      writer = csv.DictWriter(f, fieldnames=data.keys())
      if not csv_path.is_file():
        writer.writeheader()
      writer.writerow(data)
  # Process predictions
  for i, det in enumerate(pred):  # per image
    seen += 1
    if webcam:  # batch_size >= 1
      p, im0, frame = path[i], im0s[i].copy(), dataset.count
      s += f"{i}: "
    else:
      p, im0, frame = path, im0s.copy(), getattr(dataset, "frame", 0)
    p = Path(p)  # to Path
    save_path = str(save_dir / p.name)  # im.jpg
    txt_path = str(save_dir / "labels" / p.stem) + ("" if dataset.mode == "image" else f"_{frame}")  # im.txt
    s += "%gx%g " % im.shape[2:]  # print string
    gn = torch.tensor(im0.shape)[[1, 0, 1, 0]]  # normalization gain whwh
    imc = im0.copy() if args["save_crop"] else im0  # for save_crop
    annotator = Annotator(im0, line_width=args["line_thickness"], example=str(names))
    if len(det):
      # Rescale boxes from img_size to im0 size
      det[:, :4] = scale_boxes(im.shape[2:], det[:, :4], im0.shape).round()
      # Print results
      for c in det[:, 5].unique():
        n = (det[:, 5] == c).sum()  # detections per class
        s += f"{n} {names[int(c)]}{'s' * (n > 1)}, "  # add to string
      # Write results
      for *xyxy, conf, cls in reversed(det):
        c = int(cls)  # integer class
        label = names[c] if args["hide_conf"] else f"{names[c]}"
        confidence = float(conf)
        confidence_str = f"{confidence:.2f}"
        if args["save_csv"]:
          write_to_csv(p.name, label, confidence_str)
        if args["save_txt"]:  # Write to file
          xywh = (xyxy2xywh(torch.tensor(xyxy).view(1, 4)) / gn).view(-1).tolist()  # normalized xywh
          line = (cls, *xywh, conf) if args["save_conf"] else (cls, *xywh)  # label format
          with open(f"{txt_path}.txt", "a") as f:
            f.write(("%g " * len(line)).rstrip() % line + "\n")
        if save_img or args["save_crop"] or args["view_img"]:  # Add bbox to image
          c = int(cls)  # integer class
          label = None if args["hide_labels"] else (names[c] if args["hide_conf"] else f"{names[c]} {conf:.2f}")
          annotator.box_label(xyxy, label, color=colors(c, True))
        if args["save_crop"]:
          save_one_box(xyxy, imc, file=save_dir / "crops" / names[c] / f"{p.stem}.jpg", BGR=True)

    # Stream results
    im0 = annotator.result()
    if args["view_img"]:
      if platform.system() == "Linux" and p not in windows:
        windows.append(p)
        cv2.namedWindow(str(p), cv2.WINDOW_NORMAL | cv2.WINDOW_KEEPRATIO)  # allow window resize (Linux)
        cv2.resizeWindow(str(p), im0.shape[1], im0.shape[0])
      cv2.imshow(str(p), im0)
      cv2.waitKey(1)  # 1 millisecond

    # Save results (image with detections)
    if save_img:
      if dataset.mode == "image":
        cv2.imwrite(save_path, im0)
      else:  # 'video' or 'stream'
        if vid_path[i] != save_path:  # new video
          vid_path[i] = save_path
          if isinstance(vid_writer[i], cv2.VideoWriter):
            vid_writer[i].release()  # release previous video writer
          if vid_cap:  # video
            fps = vid_cap.get(cv2.CAP_PROP_FPS)
            w = int(vid_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(vid_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
          else:  # stream
            fps, w, h = 30, im0.shape[1], im0.shape[0]
          save_path = str(Path(save_path).with_suffix(".mp4"))  # force *.mp4 suffix on results videos
          vid_writer[i] = cv2.VideoWriter(save_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
        vid_writer[i].write(im0)

  # Print time (inference-only)
  LOGGER.info(f"{s}{'' if len(det) else '(no detections), '}{dt[1].dt * 1E3:.1f}ms")

顯示每個圖片在不同處理步驟的平均花費時間、儲存位置資訊

In [ ]:
# Print results
t = tuple(x.t / seen * 1e3 for x in dt)  # speeds per image
LOGGER.info(f"Speed: %.1fms pre-process, %.1fms inference, %.1fms NMS per image at shape {(1, 3, *imgsz)}" % t)
if args["save_txt"] or save_img:
  s = f"\n{len(list(save_dir.glob('labels/*.txt')))} labels saved to {save_dir / 'labels'}" if args["save_txt"] else ""
  LOGGER.info(f"Results saved to {colorstr('bold', save_dir)}{s}")
if args["update"]:
  strip_optimizer(weights[0])  # update model (to fix SourceChangeWarning)

顯示圖片預測結果

In [ ]:
display.Image(filename='runs/detect/exp2/bus.jpg', width=600)

# **命令列執行影片物件偵測**

Colab掛載個人drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

移動至Colab指定位置

In [ ]:
!cp /content/drive/MyDrive/20240517_yolo_colab_run_test/video.mp4 /content/yolov5/video.mp4

影片預測並將預測結果存檔

In [ ]:
!python detect.py --weights yolov5s.pt --img 640 --conf 0.25 --source video.mp4

透過ffmpeg將影片轉為libx264編碼(廣泛支援的影片編碼格式)

In [ ]:
from base64 import b64encode
import os

# Input video path
save_path = "runs/detect/exp3/video.mp4"

# Compressed video path
compressed_path = "runs/detect/exp3/video_compress.mp4"

os.system(f"ffmpeg -i {save_path} -vcodec libx264 {compressed_path}")

顯示影片預測結果

In [ ]:
# Show video
path = "runs/detect/exp3/video_compress.mp4"
mp4 = open(path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display.HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

# **僅供參考: 命令列執行驗證**

下載用來驗證的資料集

In [ ]:
# Download COCO val
torch.hub.download_url_to_file('https://ultralytics.com/assets/coco2017val.zip', 'tmp.zip')  # download (780M - 5000 images)
!unzip -q tmp.zip -d ../datasets && rm tmp.zip  # unzip

使用資料集評估模型的訓練結果 (ref. cpu: 47min)

In [ ]:
# Validate YOLOv5s on COCO val
!python val.py --weights yolov5s.pt --data coco.yaml --img 640

# **僅供參考: 命令列執行訓練**

下載訓練資料集並以yolov5s.pt為初始權重參數訓練(ref. cpu:18min)

In [ ]:
# Train YOLOv5s on COCO128 for 3 epochs
!python train.py --img 640 --batch 16 --epochs 3 --data coco128.yaml --weights yolov5s.pt --cache